In [13]:
SELECTED_FEATURES = [
    "having_IP_Address",
    "URL_Length",
    "having_At_Symbol",
    "double_slash_redirecting",
    "Prefix_Suffix",
    "having_Sub_Domain",
    "SSLfinal_State",
    "HTTPS_token",
    "Shortining_Service",
    "DNSRecord",
    "URL_of_Anchor"
]

In [1]:
# ==========================================================
# PhishGuard AI - Model Training Notebook
# Section 1: Install Required Libraries
# ==========================================================

!pip install lightgbm xgboost liac-arff scipy joblib

  Preparing metadata (setup.py) ... done
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=8e1aa025a68e002e4d4def5aedc5e7e811843040e7af7a3f7814a915966dfb88
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


In [4]:
# ==========================================================
# Section 2: Import Libraries
# ==========================================================

import pandas as pd
import numpy as np

from scipy.io import arff

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import joblib

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


In [3]:
# ==========================================================
# Section 3: Mount Google Drive
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ==========================================================
# Section 4: Dataset Path
# ==========================================================

dataset_path = "/content/drive/MyDrive/Training Dataset.arff"

print(dataset_path)

/content/drive/MyDrive/Training Dataset.arff


In [31]:
# ==========================================================
# Section 5: Load Dataset
# ==========================================================

from scipy.io import arff

data, meta = arff.loadarff(dataset_path)

df = pd.DataFrame(data)

# Decode byte strings to normal strings
for column in df.select_dtypes([object]):
    df[column] = df[column].str.decode("utf-8")

# Convert ALL columns to integers
df = df.astype(int)

print("✅ Dataset loaded successfully!")
print(f"Dataset Shape: {df.shape}")

print("\nData Types:")
print(df.dtypes)

✅ Dataset loaded successfully!
Dataset Shape: (11055, 31)

Data Types:
having_IP_Address              int64
URL_Length                     int64
Shortining_Service             int64
having_At_Symbol               int64
double_slash_redirecting       int64
Prefix_Suffix                  int64
having_Sub_Domain              int64
SSLfinal_State                 int64
Domain_registeration_length    int64
Favicon                        int64
port                           int64
HTTPS_token                    int64
Request_URL                    int64
URL_of_Anchor                  int64
Links_in_tags                  int64
SFH                            int64
Submitting_to_email            int64
Abnormal_URL                   int64
Redirect                       int64
on_mouseover                   int64
RightClick                     int64
popUpWidnow                    int64
Iframe                         int64
age_of_domain                  int64
DNSRecord                      int64
web_

In [32]:
# ==========================================================
# Section 6: Explore Dataset
# ==========================================================

print("Dataset Shape:")
print(df.shape)

print("\n")

print("Column Names:")
print(df.columns.tolist())

print("\n")

print("First Five Rows:")
display(df.head())

print("\n")

print("Missing Values:")
print(df.isnull().sum())

print("\n")

print("Class Distribution:")
print(df["Result"].value_counts())

Dataset Shape:
(11055, 31)


Column Names:
['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report', 'Result']


First Five Rows:


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,-1,1,1,1,-1,-1,-1,-1,-1,1,...,1,1,-1,-1,-1,-1,1,1,-1,-1
1,1,1,1,1,1,-1,0,1,-1,1,...,1,1,-1,-1,0,-1,1,1,1,-1
2,1,0,1,1,1,-1,-1,-1,-1,1,...,1,1,1,-1,1,-1,1,0,-1,-1
3,1,0,1,1,1,-1,-1,-1,1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
4,1,0,-1,1,1,-1,1,1,-1,1,...,-1,1,-1,-1,0,-1,1,1,1,1




Missing Values:
having_IP_Address              0
URL_Length                     0
Shortining_Service             0
having_At_Symbol               0
double_slash_redirecting       0
Prefix_Suffix                  0
having_Sub_Domain              0
SSLfinal_State                 0
Domain_registeration_length    0
Favicon                        0
port                           0
HTTPS_token                    0
Request_URL                    0
URL_of_Anchor                  0
Links_in_tags                  0
SFH                            0
Submitting_to_email            0
Abnormal_URL                   0
Redirect                       0
on_mouseover                   0
RightClick                     0
popUpWidnow                    0
Iframe                         0
age_of_domain                  0
DNSRecord                      0
web_traffic                    0
Page_Rank                      0
Google_Index                   0
Links_pointing_to_page         0
Statistical_report       

In [33]:
# ==========================================================
# Section 7: Verify Selected Features
# ==========================================================

selected_features = [
    "having_IP_Address",
    "URL_Length",
    "having_At_Symbol",
    "double_slash_redirecting",
    "Prefix_Suffix",
    "having_Sub_Domain",
    "SSLfinal_State",
    "HTTPS_token",
    "Shortining_Service",
    "DNSRecord",
    "URL_of_Anchor"
]

print("Selected Features:\n")

for feature in selected_features:
    if feature in df.columns:
        print(f"✅ {feature}")
    else:
        print(f"❌ {feature}")

Selected Features:

✅ having_IP_Address
✅ URL_Length
✅ having_At_Symbol
✅ double_slash_redirecting
✅ Prefix_Suffix
✅ having_Sub_Domain
✅ SSLfinal_State
✅ HTTPS_token
✅ Shortining_Service
✅ DNSRecord
✅ URL_of_Anchor


In [34]:
# ==========================================================
# Section 8: Select Features and Target
# ==========================================================

X = df[SELECTED_FEATURES]

y = df["Result"]

print("Selected Features:")
print(selected_features)

print("\nFeature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Selected Features:
['having_IP_Address', 'URL_Length', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'HTTPS_token', 'Shortining_Service', 'DNSRecord', 'URL_of_Anchor']

Feature Matrix Shape: (11055, 11)
Target Shape: (11055,)


In [35]:
# ==========================================================
# Section 9: Encode Target Labels
# ==========================================================

# Convert string labels to integers
y = y.astype(int)

# Convert:
# -1 (Phishing)  -> 0
#  1 (Legitimate) -> 1

y = y.map({
    -1: 0,
     1: 1
})

print("Encoded Class Distribution:")
print(y.value_counts())

print("\nData Type:")
print(y.dtype)

Encoded Class Distribution:
Result
1    6157
0    4898
Name: count, dtype: int64

Data Type:
int64


In [36]:
# ==========================================================
# Section 10: Train/Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Set :", X_train.shape)
print("Testing Set  :", X_test.shape)

Training Set : (8844, 11)
Testing Set  : (2211, 11)


In [37]:
# ==========================================================
# Section 11: Verification
# ==========================================================

print("Training Features")
display(X_train.head())

print("\nTraining Labels")

display(y_train.head())

print("\nFeature Count:", X_train.shape[1])

Training Features


,having_IP_Address,URL_Length,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,HTTPS_token,Shortining_Service,DNSRecord,URL_of_Anchor
1758,1,-1,1,1,-1,-1,1,1,1,1,0
10910,1,-1,1,1,-1,0,0,1,1,1,-1
7412,1,-1,1,1,-1,1,1,1,1,1,0
5805,1,-1,1,1,1,1,1,1,1,1,1
9259,1,1,-1,-1,1,1,-1,-1,-1,-1,1



Training Labels


,Result
1758,0
10910,0
7412,1
5805,1
9259,1



Feature Count: 11


In [38]:
print(df["Result"].dtype)
print(df["Result"].unique())

int64
[-1  1]


In [39]:
# ==========================================================
# Section 12: Train XGBoost
# ==========================================================

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

print("✅ XGBoost model trained successfully!")

✅ XGBoost model trained successfully!


In [40]:
# ==========================================================
# Section 13: Make Predictions
# ==========================================================

xgb_predictions = xgb_model.predict(X_test)

print("✅ Predictions generated.")

✅ Predictions generated.


In [46]:
# ==========================================================
# Section 14: Evaluate XGBoost
# ==========================================================

xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_precision = precision_score(y_test, xgb_predictions)
xgb_recall = recall_score(y_test, xgb_predictions)
xgb_f1 = f1_score(y_test, xgb_predictions)

print("=" * 40)
print("XGBoost Performance")
print("=" * 40)

print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")

XGBoost Performance
Accuracy : 0.9335
Precision: 0.9254
Recall   : 0.9578
F1 Score : 0.9413


In [47]:
# ==========================================================
# Section 15: Confusion Matrix
# ==========================================================

cm = confusion_matrix(y_test, xgb_predictions)

print("Confusion Matrix:")

print(cm)

print("\nClassification Report:\n")

print(classification_report(y_test, xgb_predictions))

Confusion Matrix:
[[ 885   95]
 [  52 1179]]

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.90      0.92       980
           1       0.93      0.96      0.94      1231

    accuracy                           0.93      2211
   macro avg       0.93      0.93      0.93      2211
weighted avg       0.93      0.93      0.93      2211



In [48]:
# ==========================================================
# Section 16: Save XGBoost Model
# ==========================================================

joblib.dump(xgb_model, "xgboost.pkl")

print("✅ Model saved as xgboost.pkl")

✅ Model saved as xgboost.pkl


In [49]:
# ==========================================================
# Section 17: Download Model
# ==========================================================

from google.colab import files

files.download("xgboost.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [51]:
# ==========================================================
# Section 18: Train LightGBM
# ==========================================================

lgbm_model = LGBMClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

print("✅ LightGBM model trained successfully!")

[LightGBM] [Info] Number of positive: 4926, number of negative: 3918
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 33
[LightGBM] [Info] Number of data points in the train set: 8844, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.556988 -> initscore=0.228946
[LightGBM] [Info] Start training from score 0.228946
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [52]:
# ==========================================================
# Section 19: Make Predictions
# ==========================================================

lgbm_predictions = lgbm_model.predict(X_test)

print("✅ Predictions generated.")

✅ Predictions generated.


In [53]:
# ==========================================================
# Section 20: Evaluate LightGBM
# ==========================================================

lgbm_accuracy = accuracy_score(y_test, lgbm_predictions)
lgbm_precision = precision_score(y_test, lgbm_predictions)
lgbm_recall = recall_score(y_test, lgbm_predictions)
lgbm_f1 = f1_score(y_test, lgbm_predictions)

print("=" * 40)
print("LightGBM Performance")
print("=" * 40)

print(f"Accuracy : {lgbm_accuracy:.4f}")
print(f"Precision: {lgbm_precision:.4f}")
print(f"Recall   : {lgbm_recall:.4f}")
print(f"F1 Score : {lgbm_f1:.4f}")

LightGBM Performance
Accuracy : 0.9317
Precision: 0.9225
Recall   : 0.9578
F1 Score : 0.9398


In [54]:
# ==========================================================
# Section 21: LightGBM Confusion Matrix
# ==========================================================

lgbm_cm = confusion_matrix(y_test, lgbm_predictions)

print("Confusion Matrix:")
print(lgbm_cm)

print("\nClassification Report:\n")

print(classification_report(y_test, lgbm_predictions))

Confusion Matrix:
[[ 881   99]
 [  52 1179]]

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.90      0.92       980
           1       0.92      0.96      0.94      1231

    accuracy                           0.93      2211
   macro avg       0.93      0.93      0.93      2211
weighted avg       0.93      0.93      0.93      2211



In [55]:
# ==========================================================
# Section 22: Save LightGBM Model
# ==========================================================

joblib.dump(lgbm_model, "lightgbm.pkl")

print("✅ LightGBM model saved successfully!")

✅ LightGBM model saved successfully!


In [56]:
# ==========================================================
# Section 23: Download LightGBM Model
# ==========================================================

from google.colab import files

files.download("lightgbm.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
# ==========================================================
# Section 24: Compare Both Models
# ==========================================================

comparison = pd.DataFrame({
    "Model": ["XGBoost", "LightGBM"],
    "Accuracy": [xgb_accuracy, lgbm_accuracy],
    "Precision": [xgb_precision, lgbm_precision],
    "Recall": [xgb_recall, lgbm_recall],
    "F1 Score": [xgb_f1, lgbm_f1]
})

comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,XGBoost,0.933514,0.925432,0.957758,0.941317
1,LightGBM,0.931705,0.922535,0.957758,0.939817
